## 세포라 단일 상품 리뷰데이터 추출코드

In [ ]:
import requests
import pandas as pd
import time
import json

def get_sephora_reviews(product_id, max_pages=5):
    all_reviews = []
    limit = 100  # API 효율을 위해 100으로 설정
    passkey = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
    
    for page in range(max_pages):
        offset = page * limit
        url = "https://api.bazaarvoice.com/data/reviews.json"
        
        params = {
            "Filter": [f"contentlocale:en*", f"ProductId:{product_id}"],
            "Sort": "SubmissionTime:desc",
            "Limit": limit,
            "Offset": offset,
            "Include": "Products,Comments",
            "Stats": "Reviews",
            "passkey": passkey,
            "apiversion": "5.4",
            "Locale": "en_US"
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            reviews = data.get('Results', [])
            
            if not reviews:
                print(f"\n✅ 모든 리뷰 수집 완료 (총 {len(all_reviews)}개)")
                break
                
            for rev in reviews:
                # 데이터 추출
                review_data = {
                    "ReviewId": rev.get("Id"),
                    "Author": rev.get("UserNickname"),
                    "Rating": rev.get("Rating"),
                    "Title": rev.get("Title"),
                    "ReviewText": rev.get("ReviewText"),
                    "SubmissionTime": rev.get("SubmissionTime"),
                    "IsRecommended": rev.get("IsRecommended"),
                    # 인센티브 여부 안전하게 추출
                    "Incentivized": rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A"),
                    "Helpfulness": rev.get("Helpfulness")
                }
                all_reviews.append(review_data)
            
            print(f"Page {page+1} 수집 중... (현재 {len(all_reviews)}개)", end='\r')
            time.sleep(0.5)
            
        except Exception as e:
            print(f"\n❌ 에러 발생: {e}")
            break
            
    # --- JSON 파일 저장 (리스트 형태 직접 저장) ---
    filename = f"sephora_test_crawling.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_reviews, f, ensure_ascii=False, indent=4)
    print(f"\n📂 '{filename}' 저장 완료!")
    
    return pd.DataFrame(all_reviews)

# --- 실행 부분 ---
PRODUCT_ID = "P514736"
df = get_sephora_reviews(PRODUCT_ID, max_pages=10)

# 데이터프레임 상위 5개 확인
if not df.empty:
    display(df.head())